# MOSCOT PFC dataset training

Set STP_PROJECT before launching Jupyter. The bundle README lists inputs, kernels and run order. Run the sections for your workflow.

In [ ]:
import os
from pathlib import Path

# Set these before starting Jupyter. Existing per-notebook overrides still apply.
if "STP_PROJECT" not in os.environ:
    raise RuntimeError("Set STP_PROJECT to the experiment workspace before starting Jupyter.")
project_root = Path(os.environ["STP_PROJECT"]).expanduser().resolve()
os.environ["STP_PROJECT"] = str(project_root)
os.environ.setdefault("SPATIAL_TI_PROJECT", str(project_root))
os.environ.setdefault("PFC_PROJECT", str(project_root))
os.environ.setdefault("FLOW_STP_REPO", str(project_root))
os.environ.setdefault("QIAN_PFC_DATA_ROOT", str(project_root / "data/qian_pfc"))
os.environ.setdefault("QIAN_DATA_ROOT", os.environ["QIAN_PFC_DATA_ROOT"])
os.environ.setdefault("PFC_DATA_ROOT", os.environ["QIAN_PFC_DATA_ROOT"])
os.environ.setdefault("STVCR_REPO", str(project_root / "external/stVCR"))

## Fit the three-age spatial model

In [ ]:
# ===========================================================================
# QIAN PFC | MOSCOT SPATIOTEMPORAL — AUTHOR-TUTORIAL FAITHFUL GPU TRAINER
# ONE STANDALONE JUPYTER CELL.  FULL DATA; NOT A GW20 HOLDOUT.
# Section IDs 1 (GW15 UMB1117/F1b), 2 (GW20 FB080/F1), 7 (GW22 FB123/F2)
# Source: https://moscot.readthedocs.io/en/latest/notebooks/tutorials/500_spatiotemporal.html
# Model: SpatioTemporalProblem; local-pca(30) + spatial FGW, alpha=0.5,
# epsilon=0, rank=200. No dense NxM transport matrix is ever constructed.
# Dataset-specific adaptation: uniform marginals (human targeted 300 genes
# cannot justify the tutorial's MOUSE proliferation/apoptosis marker scoring).
# Requires moscot==0.5.2 and GPU-capable JAX in a fresh kernel.
# ===========================================================================

import os, sys
SMOKE = False  # True -> independent 1,500-cell-per-age feasibility test.
SEED = 20261024
if "jax" in sys.modules or "moscot" in sys.modules:
    raise RuntimeError("Restart this Jupyter kernel before running: JAX/moscot already imported")
os.environ["JAX_PLATFORMS"] = "cuda"  # NVIDIA only, avoid unavailable ROCm backend
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"  # GPU VRAM on demand
# No host-RAM cap or GPU memory reservation. Slurm allocation still applies.
try:
    N_CPUS = len(os.sched_getaffinity(0))
except (OSError, AttributeError):
    N_CPUS = os.cpu_count() or 1
N_CPUS = max(1, min(N_CPUS, int(os.environ.get("SLURM_CPUS_PER_TASK") or N_CPUS)))
for _v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS", "NUMEXPR_NUM_THREADS"):
    os.environ[_v] = str(N_CPUS)

from pathlib import Path
import gc, hashlib, inspect, json, platform, random, subprocess, time
from importlib.metadata import version
import numpy as np
import pandas as pd
import scipy.sparse as sp
import scanpy as sc
import anndata as ad
import jax
import moscot
from moscot.problems.spatiotemporal import SpatioTemporalProblem

random.seed(SEED)
np.random.seed(SEED)
DEVICES = jax.devices()
assert DEVICES and all(d.platform == "gpu" for d in DEVICES), f"Require NVIDIA CUDA: {DEVICES}"
assert version("moscot") == "0.5.2", f"Version mismatch: {version('moscot')}; please use original moscot 0.5.2 env"
print("Moscot", version("moscot"), "OTT", version("ott-jax"), "JAX", version("jax"), flush=True)
print("GPU:", DEVICES, "| JAX allocation: on-demand | CPUs:", N_CPUS, flush=True)
try:
    result = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,memory.used", "--format=csv,noheader"],
                            capture_output=True, text=True, timeout=8, check=False)
    if result.returncode == 0: print("GPU VRAM:", result.stdout.strip(), flush=True)
except (FileNotFoundError, subprocess.TimeoutExpired):
    pass

PROJECT = Path(os.environ["STP_PROJECT"])
DATA_ROOT = Path(os.environ["QIAN_PFC_DATA_ROOT"])
OUT = (PROJECT / "experiments/qian_pfc_publication_4way_20261024"
       / ("moscot_slices_1_2_7_AUTHOR_TUTORIAL_GPU_SMOKE" if SMOKE else
          "moscot_slices_1_2_7_AUTHOR_TUTORIAL_GPU"))
INPUT = OUT / "moscot_input_3sections.h5ad"
EXPECTED = [(1, "GW15", "UMB1117", "F1b", 410492, 15.0),
            (2, "GW20", "FB080", "F1", 665204, 20.0),
            (7, "GW22", "FB123", "F2", 439986, 22.0)]
PAIR_NAMES = {(15.0, 20.0): "GW15_to_GW20", (20.0, 22.0): "GW20_to_GW22"}
assert DATA_ROOT.is_dir(), DATA_ROOT

def save_json(path, data):
    path.write_text(json.dumps(data, indent=2, sort_keys=True, default=str))

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(8 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

if (OUT / "FIT_COMPLETE.json").exists():
    print("Previous completed run:", OUT, "(no overwrite/retraining).")
elif OUT.exists() and any(OUT.iterdir()):
    raise RuntimeError(f"Directory has partial artifacts: {OUT}. Inspect these before running; don't overwrite a previous attempt.")
else:
    OUT.mkdir(parents=True, exist_ok=True)
    sys.path.insert(0, str(PROJECT / "src"))
    from flow_stp.data.qian_pfc import QianPFCConfig, build_qian_pfc_panel, read_h5ad_rows_direct

    # SAME selected sections as the successful stVCR/STORIES full-data runs.
    cfg = QianPFCConfig(
        data_root=DATA_ROOT,
        targets=(("GW15", ("F1a", "F1b")),
                 ("GW20", ("F1", "F2a", "F2b")),
                 ("GW22", ("F1", "F2", "F3"))),
        expected_num_slices=9, layer_key_for_counts="raw")
    inventory = build_qian_pfc_panel(cfg)
    exp_inventory = [("GW15", "UMB1117", "F1a"), ("GW15", "UMB1117", "F1b"),
                     ("GW20", "FB080", "F1"), ("GW20", "FB080", "F2a"),
                     ("GW20", "FB080", "F2b"), ("GW20", "FB121", "F1"),
                     ("GW22", "FB123", "F1"), ("GW22", "FB123", "F2"),
                     ("GW22", "FB123", "F3")]
    assert [(str(r.gw), str(r["sample"]), str(r.region)) for _, r in inventory.iterrows()] == exp_inventory
    selected = inventory.loc[inventory.slice_id.isin([1, 2, 7])].copy().reset_index(drop=True)
    assert list(selected.slice_id) == [1, 2, 7]
    for (_, r), (sid, gw, sample, region, n, _) in zip(selected.iterrows(), EXPECTED):
        assert (int(r.slice_id), str(r.gw), str(r["sample"]), str(r.region), int(r.n_full)) == (sid,gw,sample,region,n)
    inventory.to_csv(OUT / "all_nine_inventory.csv", index=False)
    selected.to_csv(OUT / "selected_three_tissues.csv", index=False)
    print("\nExact three-section cohort:\n", selected[["slice_id", "gw", "sample", "region", "n_full"]].to_string(index=False), flush=True)

    count_parts, xy_parts, obs_parts = [], [], []
    genes0 = None
    for sid, gw, sample, region, n_full, week in EXPECTED:
        r = selected.loc[selected.slice_id == sid].iloc[0]
        path = Path(r.path)
        backed = sc.read_h5ad(path, backed="r")
        try:
            assert backed.raw is not None, f"No raw count matrix in {path}"
            genes = list(map(str, backed.raw.var_names))
            assert len(genes) == 300 and len(set(genes)) == 300
            if genes0 is None: genes0 = genes
            else: assert genes == genes0, "Gene ordering mismatch"
            mask = ((backed.obs["sample"].astype(str).to_numpy() == sample) &
                    (backed.obs["region"].astype(str).to_numpy() == region))
            indices = np.flatnonzero(mask)
            assert len(indices) == n_full, (sid, len(indices), n_full)
            if SMOKE:
                indices = np.sort(np.random.default_rng(SEED + sid).choice(indices, size=1500, replace=False))
            xy = np.asarray(backed.obsm["spatial"], dtype=np.float32)[indices, :2]
            cell_ids = np.asarray(backed.obs_names.astype(str))[indices]
            assert np.isfinite(xy).all() and xy.shape == (len(indices), 2)
            extra = {}
            for label in ("H1_annotation", "H2_annotation", "H3_annotation"):
                if label in backed.obs.columns:
                    extra[label] = backed.obs.iloc[indices][label].astype(str).to_numpy(dtype=object)
        finally:
            backed.file.close()
        raw = np.asarray(read_h5ad_rows_direct(path, indices, layer_key="raw"), dtype=np.float32)
        assert raw.shape == (len(indices), 300) and np.isfinite(raw).all()
        assert np.min(raw) >= 0 and np.allclose(raw[:min(len(raw),10000)], np.rint(raw[:min(len(raw),10000)])), "Not raw counts"
        count_parts.append(sp.csr_matrix(raw))
        xy_parts.append(xy)
        obs_parts.append(pd.DataFrame({
            "gw": gw, "sample": sample, "region": region,
            "original_slice_id": sid, "cell_id": cell_ids.astype(object),
            "time": week, **extra
        }))
        del raw, indices
        gc.collect()
        print(f"Loaded GW{int(week)} {sample}/{region}: {len(xy):,} cells", flush=True)

    obs = pd.concat(obs_parts, ignore_index=True)
    obs.index = pd.Index(np.asarray([f"s{sid}:{cid}" for sid, cid in zip(obs.original_slice_id, obs.cell_id)], dtype=object),
                         name="cell_key")
    assert obs.index.is_unique
    var = pd.DataFrame(index=pd.Index(np.asarray(genes0, dtype=object), name="gene"))
    adata = ad.AnnData(X=sp.vstack(count_parts, format="csr"), obs=obs, var=var)
    adata.obsm["spatial"] = np.vstack(xy_parts).astype(np.float32)
    del count_parts, xy_parts, obs_parts, obs
    gc.collect()
    assert adata.n_obs == (4500 if SMOKE else 1515682) and adata.n_vars == 300
    assert adata.obs.groupby("time", observed=True).size().to_dict() == {
        15.0: 1500 if SMOKE else 410492,
        20.0: 1500 if SMOKE else 665204,
        22.0: 1500 if SMOKE else 439986}

    # Tutorial inputs are normalized and log-transformed counts. Preserve all
    # 300 common genes. NO 20D external PCA: the author's local-pca callback
    # independently fits joint PCA30 for EACH consecutive timepoint pair.
    print("Preprocessing exact raw counts -> logCP10k ...", flush=True)
    sc.pp.normalize_total(adata, target_sum=10000.0)
    sc.pp.log1p(adata)
    if sp.issparse(adata.X):
        adata.X = adata.X.tocsr().astype(np.float32)
        assert np.isfinite(adata.X.data).all() and np.min(adata.X.data, initial=0) >= 0
    else:
        adata.X = np.asarray(adata.X, dtype=np.float32)
        assert np.isfinite(adata.X).all() and (adata.X >= 0).all()
    # Ensure string-compatible HDF5 storage across pandas/anndata versions.
    for df in (adata.obs, adata.var):
        df.index = pd.Index(np.asarray(df.index.tolist(), dtype=object), name=df.index.name)
        for column in df.columns:
            if isinstance(df[column].dtype, pd.StringDtype):
                df[column] = df[column].astype(object)
    print("Saving logged input:", INPUT, flush=True)
    adata.write_h5ad(INPUT, compression="gzip")

    # Freeze methodological choices BEFORE fitting.
    run_config = {
        "experiment": "moscot full-data Qian PFC matched 3 ages; NOT strict GW20 held-out",
        "tutorial": "https://moscot.readthedocs.io/en/latest/notebooks/tutorials/500_spatiotemporal.html",
        "moscot": version("moscot"), "jax": version("jax"), "ott_jax": version("ott-jax"),
        "training_original_section_ids": [1, 2, 7],
        "selected_three_tissues": [(sid, gw, sample, region, n) for sid,gw,sample,region,n,_ in EXPECTED],
        "n_cells": adata.n_obs, "n_genes": adata.n_vars,
        "times": [15., 20., 22.], "seed": SEED, "smoke": SMOKE,
        "raw_counts": "source H5AD /raw/X; no gene selection",
        "expression_preprocessing": "Normalize raw counts per cell to 10000, then log1p",
        "gene_cost_representation": "30-component local PCA fitted separately per adjacent age pair via xy_callback='local-pca'",
        "geometry": "Original observed XY; normalized within each problem via normalize_spatial=True",
        "prepare": {"time_key":"time", "spatial_key":"spatial", "joint_attr":None,
                    "xy_callback":"local-pca", "normalize_spatial":True,
                    "policy":"sequential", "a":False, "b":False},
        "solve": {"alpha":0.5, "epsilon":0., "rank":200},
        "marginals": "Uniform; tutorial mouse proliferation/apoptosis gene sets NOT used for human targeted 300-gene data",
        "jax_devices": [str(d) for d in DEVICES], "cpus": N_CPUS,
        "data_root":str(DATA_ROOT), "output":str(OUT), "host":platform.node(),
        "note": "Reproduce tutorial's mathematical model and solver; uniform marginals are the one biological adaptation."
    }
    save_json(OUT / "run_config.json", run_config)

    # This is the version-pinned 0.5.2 API. Tutorial HTML uses older generic
    # `callback=` argument; named xy_callback is the supported equivalent.
    assert "xy_callback" in inspect.signature(SpatioTemporalProblem.prepare).parameters
    print("\nPreparing low-rank FGW pair problems with the official local PCA...", flush=True)
    stp = SpatioTemporalProblem(adata).prepare(
        time_key="time", spatial_key="spatial", joint_attr=None,
        xy_callback="local-pca", normalize_spatial=True, policy="sequential",
        a=False, b=False,   # uniform marginals; no unusable mouse gene scoring
    )
    assert set(stp.problems) == {(15.0, 20.0), (20.0, 22.0)}, list(stp.problems)
    for pair, problem in stp.problems.items():
        print("Pair", pair, "problem shape:", problem.shape, flush=True)

    print("\nSolving with the authors' parameters: alpha=.5, epsilon=0, rank=200", flush=True)
    tic = time.perf_counter()
    stp = stp.solve(alpha=0.5, epsilon=0.0, rank=200)
    fit_seconds = time.perf_counter() - tic
    print("Solver returned after", round(fit_seconds, 2), "seconds", flush=True)
    assert set(stp.solutions) == set(PAIR_NAMES), list(stp.solutions)

    # Save official native LOW RANK plan Q diag(1/g) R^T. Dense coupling would
    # require >1 TB for first PFC age pair and must not be constructed.
    # Access `_output` is version-specific and intentionally recorded.
    records = []
    for pair in sorted(stp.solutions):
        sol = stp.solutions[pair]
        src, tgt = map(float, pair)
        name = PAIR_NAMES[(src, tgt)]
        assert bool(sol.is_low_rank), f"Unexpected full-rank solution for {pair}"
        out_native = getattr(sol, "_output", None)
        if out_native is None or not all(hasattr(out_native, k) for k in ("q", "r", "g")):
            raise RuntimeError(f"Version-specific low-rank OTT factors Q,R,g inaccessible for {pair}; no dense conversion attempted")
        q, r, g = (np.asarray(jax.device_get(getattr(out_native, k))) for k in ("q", "r", "g"))
        n_src = int((adata.obs["time"].to_numpy() == src).sum())
        n_tgt = int((adata.obs["time"].to_numpy() == tgt).sum())
        assert q.shape == (n_src, 200) and r.shape == (n_tgt, 200) and g.shape == (200,), (q.shape,r.shape,g.shape)
        assert all(np.isfinite(m).all() for m in (q,r,g)) and np.all(g > 0)
        pair_dir = OUT / name
        pair_dir.mkdir(exist_ok=False)
        np.save(pair_dir / "Q.npy", q)
        np.save(pair_dir / "R.npy", r)
        np.save(pair_dir / "g.npy", g)
        np.save(pair_dir / "source_obs_positions.npy", np.flatnonzero(adata.obs.time.to_numpy() == src).astype(np.int64))
        np.save(pair_dir / "target_obs_positions.npy", np.flatnonzero(adata.obs.time.to_numpy() == tgt).astype(np.int64))
        info = {
            "source": src, "target": tgt, "pair": name, "shape": [n_src, n_tgt],
            "rank": 200, "converged": bool(sol.converged), "regularized_cost": float(sol.cost),
            "factor_formula": "T = Q @ diag(1/g) @ R.T",
            "native_factor_access": "moscot OTTOutput._output (moscot 0.5.2)",
            "factor_dtypes": [str(z.dtype) for z in (q,r,g)],
            "source_factor_mass": float(q.sum(dtype=np.float64)),
            "target_factor_mass": float(r.sum(dtype=np.float64)),
            "sha256": {p.name: sha256(p) for p in sorted(pair_dir.glob("*.npy"))},
        }
        save_json(pair_dir / "solution_manifest.json", info)
        records.append(info)
        print(name, "converged", sol.converged, "cost", sol.cost,
              "factor sizes", (q.shape,r.shape,g.shape), flush=True)
        del q, r, g
        gc.collect()

    complete = {"status": "completed" if all(d["converged"] for d in records)
                else "completed_with_nonconvergence",
                "fit_seconds": fit_seconds, "pairs": records,
                "run_config_sha256": sha256(OUT / "run_config.json"),
                "input_sha256": sha256(INPUT)}
    save_json(OUT / "FIT_COMPLETE.json", complete)
    if all(d["converged"] for d in records):
        save_json(OUT / "SUCCESS.json", complete)
        print("\nMOSCOT FULL-DATA FIT COMPLETED; BOTH SOLVERS REPORT CONVERGENCE.", flush=True)
    else:
        print("\nMOSCOT FULL-DATA FIT COMPLETED; NONCONVERGENCE REPORTED.", flush=True)
        print("The authors' own tutorial also reports nonconvergence. Preserve factors for QC; do not mark publication-ready yet.", flush=True)
    print("Artifacts:", OUT, flush=True)

## Fit the strict GW20 holdout model

In [ ]:
# ===========================================================================
# QIAN PFC | MOSCOT SPATIOTEMPORAL — STRICT GW20 HOLDOUT GPU TRAINER
# ONE STANDALONE JUPYTER CELL. Train ONLY on GW15 and GW22; GW20 is NEVER read.
# Original nine-panel section IDs: 1 (GW15 UMB1117/F1b), 7 (GW22 FB123/F2).
# This is a NEW separate fit, not a reuse of the 3-age model.
# Source: https://moscot.readthedocs.io/en/latest/notebooks/tutorials/500_spatiotemporal.html
# Model: SpatioTemporalProblem; local-pca(30) + spatial FGW, alpha=0.5,
# epsilon=0, rank=200. No dense NxM transport matrix is ever constructed.
# Dataset-specific adaptation: uniform marginals (human targeted 300 genes
# cannot justify the tutorial's MOUSE proliferation/apoptosis marker scoring).
# Requires moscot==0.5.2 and GPU-capable JAX in a fresh kernel.
# A two-age FGW coupling does NOT natively predict an unobserved intermediate age;
# GW20 evaluation will require a separately specified, frozen prediction rule.
# ===========================================================================

import os, sys
SMOKE = False  # True -> independent 1,500-cell-per-age feasibility test.
SEED = 20261024  # Same random seed as the successful three-age run.
if "jax" in sys.modules or "moscot" in sys.modules:
    raise RuntimeError("Restart this Jupyter kernel before running: JAX/moscot already imported")
os.environ["JAX_PLATFORMS"] = "cuda"  # NVIDIA only, avoid unavailable ROCm backend
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"  # GPU VRAM on demand
# No host-RAM cap or GPU memory reservation. Slurm allocation still applies.
try:
    N_CPUS = len(os.sched_getaffinity(0))
except (OSError, AttributeError):
    N_CPUS = os.cpu_count() or 1
N_CPUS = max(1, min(N_CPUS, int(os.environ.get("SLURM_CPUS_PER_TASK") or N_CPUS)))
for _v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS", "NUMEXPR_NUM_THREADS"):
    os.environ[_v] = str(N_CPUS)

from pathlib import Path
import gc, hashlib, inspect, json, platform, random, subprocess, time
from importlib.metadata import version
import numpy as np
import pandas as pd
import scipy.sparse as sp
import scanpy as sc
import anndata as ad
import jax
import moscot
from moscot.problems.spatiotemporal import SpatioTemporalProblem

random.seed(SEED)
np.random.seed(SEED)
DEVICES = jax.devices()
assert DEVICES and all(d.platform == "gpu" for d in DEVICES), f"Require NVIDIA CUDA: {DEVICES}"
assert version("moscot") == "0.5.2", f"Version mismatch: {version('moscot')}; please use original moscot 0.5.2 env"
print("Moscot", version("moscot"), "OTT", version("ott-jax"), "JAX", version("jax"), flush=True)
print("GPU:", DEVICES, "| JAX allocation: on-demand | CPUs:", N_CPUS, flush=True)
try:
    result = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,memory.used", "--format=csv,noheader"],
                            capture_output=True, text=True, timeout=8, check=False)
    if result.returncode == 0: print("GPU VRAM:", result.stdout.strip(), flush=True)
except (FileNotFoundError, subprocess.TimeoutExpired):
    pass

PROJECT = Path(os.environ["STP_PROJECT"])
DATA_ROOT = Path(os.environ["QIAN_PFC_DATA_ROOT"])
OUT = (PROJECT / "experiments/qian_pfc_publication_4way_20261024"
       / ("moscot_gw15_gw22_STRICT_GW20_HELDOUT_AUTHOR_TUTORIAL_GPU_SMOKE" if SMOKE else
          "moscot_gw15_gw22_STRICT_GW20_HELDOUT_AUTHOR_TUTORIAL_GPU"))
INPUT = OUT / "moscot_input_gw15_gw22_train_only.h5ad"
EXPECTED = [(1, "GW15", "UMB1117", "F1b", 410492, 15.0),
            (7, "GW22", "FB123", "F2", 439986, 22.0)]
PAIR_NAMES = {(15.0, 22.0): "GW15_to_GW22"}
assert DATA_ROOT.is_dir(), DATA_ROOT

def save_json(path, data):
    path.write_text(json.dumps(data, indent=2, sort_keys=True, default=str))

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(8 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

if (OUT / "FIT_COMPLETE.json").exists():
    print("Previous completed run:", OUT, "(no overwrite/retraining).")
elif OUT.exists() and any(OUT.iterdir()):
    raise RuntimeError(f"Directory has partial artifacts: {OUT}. Inspect these before running; don't overwrite a previous attempt.")
else:
    OUT.mkdir(parents=True, exist_ok=True)
    sys.path.insert(0, str(PROJECT / "src"))
    from flow_stp.data.qian_pfc import QianPFCConfig, build_qian_pfc_panel, read_h5ad_rows_direct

    # STRICT AGE HOLDOUT: build_qian_pfc_panel opens only GW15 and GW22.
    # Even the training inventory must never access any GW20 source H5AD.
    cfg = QianPFCConfig(
        data_root=DATA_ROOT,
        targets=(("GW15", ("F1b",)), ("GW22", ("F2",))),
        expected_num_slices=2, layer_key_for_counts="raw")
    inventory = build_qian_pfc_panel(cfg)
    assert [(str(r.gw), str(r["sample"]), str(r.region)) for _, r in inventory.iterrows()] == [
        ("GW15", "UMB1117", "F1b"), ("GW22", "FB123", "F2")]
    selected = inventory.copy().reset_index(drop=True)
    selected["local_panel_slice_id"] = selected["slice_id"]
    selected["slice_id"] = [1, 7]  # Preserve IDs of the original nine-section benchmark.
    assert selected.slice_id.tolist() == [1, 7]
    for (_, r), (sid, gw, sample, region, n, _) in zip(selected.iterrows(), EXPECTED):
        assert (int(r.slice_id), str(r.gw), str(r["sample"]), str(r.region), int(r.n_full)) == (sid,gw,sample,region,n)
        assert Path(r.path).name == gw.lower() + ".h5ad", r.path
    assert sum(int(r.n_full) for _,r in selected.iterrows()) == 850478
    inventory.to_csv(OUT / "training_only_two_section_inventory.csv", index=False)
    selected.to_csv(OUT / "selected_two_tissues.csv", index=False)
    print("STRICT GW20 HOLDOUT: no GW20 source file or metadata was opened.", flush=True)
    print("\nExact training-only cohort:\n", selected[["slice_id", "gw", "sample", "region", "n_full"]].to_string(index=False), flush=True)

    count_parts, xy_parts, obs_parts = [], [], []
    genes0 = None
    for sid, gw, sample, region, n_full, week in EXPECTED:
        r = selected.loc[selected.slice_id == sid].iloc[0]
        path = Path(r.path)
        backed = sc.read_h5ad(path, backed="r")
        try:
            assert backed.raw is not None, f"No raw count matrix in {path}"
            genes = list(map(str, backed.raw.var_names))
            assert len(genes) == 300 and len(set(genes)) == 300
            if genes0 is None: genes0 = genes
            else: assert genes == genes0, "Gene ordering mismatch"
            mask = ((backed.obs["sample"].astype(str).to_numpy() == sample) &
                    (backed.obs["region"].astype(str).to_numpy() == region))
            indices = np.flatnonzero(mask)
            assert len(indices) == n_full, (sid, len(indices), n_full)
            if SMOKE:
                indices = np.sort(np.random.default_rng(SEED + sid).choice(indices, size=1500, replace=False))
            xy = np.asarray(backed.obsm["spatial"], dtype=np.float32)[indices, :2]
            cell_ids = np.asarray(backed.obs_names.astype(str))[indices]
            assert np.isfinite(xy).all() and xy.shape == (len(indices), 2)
            extra = {}
            for label in ("H1_annotation", "H2_annotation", "H3_annotation"):
                if label in backed.obs.columns:
                    extra[label] = backed.obs.iloc[indices][label].astype(str).to_numpy(dtype=object)
        finally:
            backed.file.close()
        raw = np.asarray(read_h5ad_rows_direct(path, indices, layer_key="raw"), dtype=np.float32)
        assert raw.shape == (len(indices), 300) and np.isfinite(raw).all()
        assert np.min(raw) >= 0 and np.allclose(raw[:min(len(raw),10000)], np.rint(raw[:min(len(raw),10000)])), "Not raw counts"
        count_parts.append(sp.csr_matrix(raw))
        xy_parts.append(xy)
        obs_parts.append(pd.DataFrame({
            "gw": gw, "sample": sample, "region": region,
            "original_slice_id": sid, "cell_id": cell_ids.astype(object),
            "time": week, **extra
        }))
        del raw, indices
        gc.collect()
        print(f"Loaded GW{int(week)} {sample}/{region}: {len(xy):,} cells", flush=True)

    obs = pd.concat(obs_parts, ignore_index=True)
    obs.index = pd.Index(np.asarray([f"s{sid}:{cid}" for sid, cid in zip(obs.original_slice_id, obs.cell_id)], dtype=object),
                         name="cell_key")
    assert obs.index.is_unique
    var = pd.DataFrame(index=pd.Index(np.asarray(genes0, dtype=object), name="gene"))
    adata = ad.AnnData(X=sp.vstack(count_parts, format="csr"), obs=obs, var=var)
    adata.obsm["spatial"] = np.vstack(xy_parts).astype(np.float32)
    del count_parts, xy_parts, obs_parts, obs
    gc.collect()
    assert adata.n_obs == (3000 if SMOKE else 850478) and adata.n_vars == 300
    assert adata.obs.groupby("time", observed=True).size().to_dict() == {
        15.0: 1500 if SMOKE else 410492,
        22.0: 1500 if SMOKE else 439986}
    assert set(adata.obs.gw.astype(str)) == {"GW15", "GW22"}
    assert set(adata.obs.original_slice_id.astype(int)) == {1, 7}
    # This assertion is about actual loaded training data, not just the requested section names.
    assert set(map(float, adata.obs.time.unique())) == {15.0, 22.0}

    # Tutorial inputs are normalized and log-transformed counts. Preserve all
    # 300 common genes. NO external/global PCA: the author's local-pca callback
    # fits the 30-PC cross-age embedding using ONLY the two training sections.
    print("Preprocessing exact raw counts -> logCP10k ...", flush=True)
    sc.pp.normalize_total(adata, target_sum=10000.0)
    sc.pp.log1p(adata)
    if sp.issparse(adata.X):
        adata.X = adata.X.tocsr().astype(np.float32)
        assert np.isfinite(adata.X.data).all() and np.min(adata.X.data, initial=0) >= 0
    else:
        adata.X = np.asarray(adata.X, dtype=np.float32)
        assert np.isfinite(adata.X).all() and (adata.X >= 0).all()
    # Ensure string-compatible HDF5 storage across pandas/anndata versions.
    for df in (adata.obs, adata.var):
        df.index = pd.Index(np.asarray(df.index.tolist(), dtype=object), name=df.index.name)
        for column in df.columns:
            if isinstance(df[column].dtype, pd.StringDtype):
                df[column] = df[column].astype(object)
    print("Saving logged input:", INPUT, flush=True)
    adata.write_h5ad(INPUT, compression="gzip")

    # Freeze methodological choices BEFORE fitting.
    run_config = {
        "experiment": "moscot STRICT GW20 age holdout; GW15 F1b and GW22 F2 ONLY",
        "tutorial": "https://moscot.readthedocs.io/en/latest/notebooks/tutorials/500_spatiotemporal.html",
        "moscot": version("moscot"), "jax": version("jax"), "ott_jax": version("ott-jax"),
        "training_original_section_ids": [1, 7],
        "training_tissues": [(sid, gw, sample, region, n) for sid,gw,sample,region,n,_ in EXPECTED],
        "strict_holdout": True,
        "heldout_developmental_age": "GW20",
        "heldout_source_files_read_for_training": False,
        "holdout_evaluation_time": 20.0,
        "n_cells": adata.n_obs, "n_genes": adata.n_vars,
        "times": [15., 22.], "seed": SEED, "smoke": SMOKE,
        "raw_counts": "source H5AD /raw/X; no gene selection",
        "expression_preprocessing": "Normalize raw counts per cell to 10000, then log1p",
        "gene_cost_representation": "30-component local PCA fitted only on the GW15/GW22 pair via xy_callback='local-pca'",
        "geometry": "Original observed XY; normalized within each problem via normalize_spatial=True",
        "prepare": {"time_key":"time", "spatial_key":"spatial", "joint_attr":None,
                    "xy_callback":"local-pca", "normalize_spatial":True,
                    "policy":"sequential", "a":False, "b":False},
        "solve": {"alpha":0.5, "epsilon":0., "rank":200},
        "marginals": "Uniform; tutorial mouse proliferation/apoptosis gene sets NOT used for human targeted 300-gene data",
        "jax_devices": [str(d) for d in DEVICES], "cpus": N_CPUS,
        "data_root":str(DATA_ROOT), "output":str(OUT), "host":platform.node(),
        "note": "Same tutorial FGW model/solver and uniform-marginal adaptation as three-age run; only GW15/GW22 were read. The learned GW15→GW22 coupling is not a native GW20 predictor. Any intermediate-age forecast must be specified separately without looking at GW20 truth."
    }
    save_json(OUT / "run_config.json", run_config)

    # This is the version-pinned 0.5.2 API. Tutorial HTML uses older generic
    # `callback=` argument; named xy_callback is the supported equivalent.
    assert "xy_callback" in inspect.signature(SpatioTemporalProblem.prepare).parameters
    print("\nPreparing low-rank FGW pair problems with the official local PCA...", flush=True)
    stp = SpatioTemporalProblem(adata).prepare(
        time_key="time", spatial_key="spatial", joint_attr=None,
        xy_callback="local-pca", normalize_spatial=True, policy="sequential",
        a=False, b=False,   # uniform marginals; no unusable mouse gene scoring
    )
    assert set(stp.problems) == {(15.0, 22.0)}, list(stp.problems)
    for pair, problem in stp.problems.items():
        print("Pair", pair, "problem shape:", problem.shape, flush=True)

    print("\nSolving with the authors' parameters: alpha=.5, epsilon=0, rank=200", flush=True)
    tic = time.perf_counter()
    stp = stp.solve(alpha=0.5, epsilon=0.0, rank=200)
    fit_seconds = time.perf_counter() - tic
    print("Solver returned after", round(fit_seconds, 2), "seconds", flush=True)
    assert set(stp.solutions) == set(PAIR_NAMES), list(stp.solutions)

    # Save official native LOW RANK plan Q diag(1/g) R^T. Dense GW15 x GW22
    # coupling would exceed 700GB even in float32; NEVER construct it.
    # Access `_output` is version-specific and intentionally recorded.
    records = []
    for pair in sorted(stp.solutions):
        sol = stp.solutions[pair]
        src, tgt = map(float, pair)
        name = PAIR_NAMES[(src, tgt)]
        assert bool(sol.is_low_rank), f"Unexpected full-rank solution for {pair}"
        out_native = getattr(sol, "_output", None)
        if out_native is None or not all(hasattr(out_native, k) for k in ("q", "r", "g")):
            raise RuntimeError(f"Version-specific low-rank OTT factors Q,R,g inaccessible for {pair}; no dense conversion attempted")
        q, r, g = (np.asarray(jax.device_get(getattr(out_native, k))) for k in ("q", "r", "g"))
        n_src = int((adata.obs["time"].to_numpy() == src).sum())
        n_tgt = int((adata.obs["time"].to_numpy() == tgt).sum())
        assert q.shape == (n_src, 200) and r.shape == (n_tgt, 200) and g.shape == (200,), (q.shape,r.shape,g.shape)
        assert all(np.isfinite(m).all() for m in (q,r,g)) and np.all(g > 0)
        pair_dir = OUT / name
        pair_dir.mkdir(exist_ok=False)
        np.save(pair_dir / "Q.npy", q)
        np.save(pair_dir / "R.npy", r)
        np.save(pair_dir / "g.npy", g)
        np.save(pair_dir / "source_obs_positions.npy", np.flatnonzero(adata.obs.time.to_numpy() == src).astype(np.int64))
        np.save(pair_dir / "target_obs_positions.npy", np.flatnonzero(adata.obs.time.to_numpy() == tgt).astype(np.int64))
        info = {
            "source": src, "target": tgt, "pair": name, "shape": [n_src, n_tgt],
            "rank": 200, "converged": bool(sol.converged), "regularized_cost": float(sol.cost),
            "factor_formula": "T = Q @ diag(1/g) @ R.T",
            "native_factor_access": "moscot OTTOutput._output (moscot 0.5.2)",
            "factor_dtypes": [str(z.dtype) for z in (q,r,g)],
            "source_factor_mass": float(q.sum(dtype=np.float64)),
            "target_factor_mass": float(r.sum(dtype=np.float64)),
            "sha256": {p.name: sha256(p) for p in sorted(pair_dir.glob("*.npy"))},
        }
        save_json(pair_dir / "solution_manifest.json", info)
        records.append(info)
        print(name, "converged", sol.converged, "cost", sol.cost,
              "factor sizes", (q.shape,r.shape,g.shape), flush=True)
        del q, r, g
        gc.collect()

    complete = {"status": "completed" if all(d["converged"] for d in records)
                else "completed_with_nonconvergence",
                "fit_seconds": fit_seconds, "pairs": records,
                "run_config_sha256": sha256(OUT / "run_config.json"),
                "input_sha256": sha256(INPUT)}
    save_json(OUT / "FIT_COMPLETE.json", complete)
    if all(d["converged"] for d in records):
        save_json(OUT / "SUCCESS.json", complete)
        print("\nMOSCOT STRICT GW20 HOLDOUT FIT COMPLETED; GW15→GW22 SOLVER REPORTS CONVERGENCE.", flush=True)
    else:
        print("\nMOSCOT STRICT GW20 HOLDOUT FIT COMPLETED; NONCONVERGENCE REPORTED.", flush=True)
        print("Retain the native factors and diagnose QC; do not silently tune the solver.", flush=True)
    print("Artifacts:", OUT, flush=True)